# Notebook C: live 7-month forecast, every 30 m pixel

Loads the model from Notebook B and forecasts all indices for Oct 2026 to Apr 2027, as one GeoTIFF per county, index and month. Last notebook of the chain.

**Reads** (all under `F:\new_predict\VI_LSTM`)
- `model\final_model.pt` and `model\test_skill_observed_weather.csv` (Notebook B)
- `county_store\` and `_static\` (Notebooks A and A2)
- `training_dataset_clipped.npz` (Notebook A)

**Writes** `F:\PREDICTED_INDICES\` (GeoTIFFs, sample-pixel tables, plots). The resumable tile cache stays in `F:\new_predict\VI_LSTM\forecast_output\tile_cache`.

**Steps:** rebuild B's scalers and check them (1-2) → open the stores and weather (3-4) → weather anomalies for each scenario (5) → run the ensemble on the sample pixels (6-8) → run every pixel tile by tile and write the GeoTIFFs (9-10).

**Reading the results**
- `mean` is the central forecast. The p10 to p90 band is a weather-scenario range, not a confidence interval.
- The `normal` line is the forecast with no weather information.
- Trust each index and lead according to the test-skill table.
- Skill with real SEAS5 weather has not been tested.

In [ ]:
import os
# avoids OpenMP crash on Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import re, glob, json, time, calendar, warnings, hashlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
import rasterio, rasterio.crs
from affine import Affine
from rasterio.warp import transform as warp_transform
from rasterio.windows import Window

WORK_DIR   = r"F:\new_predict\VI_LSTM"            # unchanged: model, stores and dataset are read from here
DATA_NPZ   = os.path.join(WORK_DIR, "training_dataset_clipped.npz")
MODEL_PATH = os.path.join(WORK_DIR, "model", "final_model.pt")           # Notebook B
SKILL_CSV  = os.path.join(WORK_DIR, "model", "test_skill_observed_weather.csv")
STORE_DIR  = os.path.join(WORK_DIR, "county_store")                     # Notebook A: the full-area stores
MANIFEST   = os.path.join(STORE_DIR, "manifest.json")
OUT_DIR    = r"F:\CORRECTION"; os.makedirs(OUT_DIR, exist_ok=True)      # NEW: all outputs of this rerun
TIFF_DIR   = os.path.join(OUT_DIR, "geotiff")
TILE_CACHE = r"F:\CORRECTION\tile_cache"                                # NEW: separate cache, old one untouched

PIX_PER_COUNTY = 30000

# --- what to produce ---
CENTRAL = "mean"
FULL_SCENARIOS = ["mean"]
RUN_FULL_AREA = True
FORCE_RECOMPUTE = False     # fresh run: ignores the tile cache and recomputes every tile. Set back to False after this run so an interruption can resume
DELETE_TILE_CACHE = False
OFFSET_PIXELS = 20000

BIAS_CSV = None
SHRINK   = {"temp": 0.5, "rh": 0.5, "wind": 0.5, "rain": 0.5}
RECENTRE = {"temp": 0.0, "rh": 0.0, "wind": 0.0, "rain": 0.0}
ANOM_CAP = 2.0                   # cap on the standardised anomaly

N_THREADS = os.cpu_count() or 4
PREFETCH_WORKERS = 2
PREDICT_CHUNK = 16384            # pixels per model batch
SEEDS_USED = None
MAX_TILES_PER_COUNTY = None
TIFF_WORKERS = 4                 # GeoTIFFs written in parallel

CLIP_MARGIN = 0.25
PLOT_INDICES = ["NDVI", "NDMI", "SAVI"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(1, N_THREADS - PREFETCH_WORKERS) if DEVICE == "cpu" else N_THREADS)
print("device:", DEVICE, "| torch threads:", torch.get_num_threads())

## 1. Load the model and the training data

In [ ]:
art = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
cfg, norm = art["cfg"], art["norm"]
print("model trained", cfg["trained"], "| clim_end", cfg["clim_end"], "| epochs", cfg["epochs"], "| seeds", len(art["states"]))
L, H, K = cfg["L"], cfg["H"], cfg["K"]
VIS, WXN, STN, COUNTIES, MONTHS = cfg["VIS"], cfg["WXN"], cfg["STN"], cfg["COUNTIES"], cfg["MONTHS"]
WX_INPUT, PRIMARY, MIN_ENC_USABLE = cfg.get("WX_INPUT", "both"), cfg["PRIMARY"], cfg["MIN_ENC_USABLE"]      # B feeds raw + anomaly weather ("both")
K_SHRINK, ELEV_BINS, SMOOTH_CLIM, Z_CLIP = cfg["K_SHRINK"], cfg["ELEV_BINS"], cfg["SMOOTH_CLIM"], cfg["Z_CLIP"]
QA_SOURCE = cfg["QA_SOURCE"]
N_W = len(WXN) * (2 if WX_INPUT == "both" else 1)

D = np.load(DATA_NPZ, allow_pickle=False)
assert [str(v) for v in D["vis"]] == VIS and [str(m) for m in D["months"]] == MONTHS, "the dataset does not match the model (indices or months differ): use the same npz as Notebook B"
CID_all = D["county_id"].astype(int)
keep = np.sort(np.concatenate([np.flatnonzero(CID_all == c)[:PIX_PER_COUNTY] for c in range(len(COUNTIES))]))
VI, QA, WX, ST, CID = D["vi"][keep], D["qa"][keep].copy(), D["wx"][keep], D["static"][keep], CID_all[keep]
N, M, K_ = VI.shape; assert K_ == K
print(f"{N} pixels x {M} months ({MONTHS[0]} .. {MONTHS[-1]}) x {K} indices")

def find_coords(D):
    keys = set(D.files)
    for k in ("xy", "coords", "coordinates", "pixel_xy", "px_xy"):
        if k in keys and D[k].ndim == 2 and D[k].shape[1] >= 2 and D[k].shape[0] == len(CID_all):
            return D[k][:, 0].astype("float64"), D[k][:, 1].astype("float64"), k
    for kx, ky in (("x", "y"), ("lon", "lat"), ("lng", "lat"), ("easting", "northing"), ("X", "Y"), ("px_x", "px_y"), ("pix_x", "pix_y")):
        if kx in keys and ky in keys and D[kx].shape == (len(CID_all),):
            return D[kx].astype("float64"), D[ky].astype("float64"), f"{kx}/{ky}"
    return None, None, None
_x, _y, _src = find_coords(D)
if _x is None: raise RuntimeError("no pixel coordinates in the npz (keys: " + ", ".join(sorted(D.files)) + "). Run the full-area Notebook A to write training_dataset_clipped.npz.")
if "dz" not in D.files: raise RuntimeError("the npz has no 'dz' (elevation offset of each pixel); run the full-area Notebook A")
COORDS = (_x[keep], _y[keep]); DZ = D["dz"][keep].astype("float32")
print(f"pixel coordinates found in '{_src}': x {COORDS[0].min():.5g}..{COORDS[0].max():.5g}, y {COORDS[1].min():.5g}..{COORDS[1].max():.5g}")

# --- QA handling identical to Notebook B ---
has_qa = (QA == 2).any(axis=(0, 1))
if QA_SOURCE and QA_SOURCE in VIS and has_qa[VIS.index(QA_SOURCE)]:
    gf = QA[:, :, VIS.index(QA_SOURCE)] == 2
    for k, v in enumerate(VIS):
        if not has_qa[k]:
            QA[:, :, k] = np.where(gf & np.isfinite(VI[:, :, k]) & (QA[:, :, k] == 0), 2, QA[:, :, k])
USABLE = np.isfinite(VI) & ((QA == 0) | (QA == 2))
CLEAN  = np.isfinite(VI) & (QA == 0)
MIDX = {m: i for i, m in enumerate(MONTHS)}
CAL  = np.array([int(m[5:7]) - 1 for m in MONTHS])
PRI_K = VIS.index(PRIMARY)

_vs = {str(s): i for i, s in enumerate(D["vi_stat_names"])}; VST = D["vi_stats"]
_lo, _hi = VST[:, _vs["p01"]].astype("float64"), VST[:, _vs["p99"]].astype("float64")
VAL_LO, VAL_HI = _lo - CLIP_MARGIN * (_hi - _lo), _hi + CLIP_MARGIN * (_hi - _lo)

HIST = np.arange(M - L, M)
wx_hist_ok = np.isfinite(WX[:, HIST, :]).all(axis=(1, 2))
print(f"encoder window: {MONTHS[HIST[0]]} .. {MONTHS[HIST[-1]]} | pixels with complete observed weather in it: {wx_hist_ok.mean():.1%}")
if wx_hist_ok.mean() < 0.5:
    raise RuntimeError("Most pixels have no observed weather in the last 12 months. Re-run Notebook A with weather up to the last month of the dataset, then Notebook B's final fit.")

## 2. Rebuild climatology, anomalies and scalers
Same code as B's final fit. The check compares them with the values saved in the model file.

In [ ]:
def shrink_clim(s, n, g, k_shrink, smooth, axis=1):
    c = (s + k_shrink * g) / (n + k_shrink)
    if smooth:
        c = 0.25 * np.roll(c, 1, axis) + 0.5 * c + 0.25 * np.roll(c, -1, axis)
    return c.astype("float32")

def fit_climatology(V, U, CAL, CID, elev, mm, n_c, n_bins, k_shrink, smooth):
    N, M, C = V.shape
    S_ = np.zeros((N, 12, C), "float32"); C_ = np.zeros((N, 12, C), "float32")
    for cm in range(12):
        sel = np.flatnonzero((CAL == cm) & mm)
        if len(sel) == 0: continue
        u = U[:, sel, :]
        S_[:, cm, :] = np.where(u, V[:, sel, :], 0).sum(1)
        C_[:, cm, :] = u.sum(1)
    edges, bins = [], np.zeros(N, int)
    for c in range(n_c):
        ic = np.flatnonzero(CID == c)
        e = np.quantile(elev[ic], np.linspace(0, 1, n_bins + 1)[1:-1]); edges.append(e); bins[ic] = np.digitize(elev[ic], e)
    all_mean = S_.sum(0, dtype="float64") / np.maximum(C_.sum(0, dtype="float64"), 1)
    G = np.zeros((n_c, n_bins, 12, C), "float32")
    for c in range(n_c):
        ic = CID == c
        cn = C_[ic].sum(0, dtype="float64")
        cmean = np.where(cn > 0, S_[ic].sum(0, dtype="float64") / np.maximum(cn, 1), all_mean)
        for b in range(n_bins):
            ib = ic & (bins == b)
            nb_, sb = C_[ib].sum(0, dtype="float64"), S_[ib].sum(0, dtype="float64")
            G[c, b] = np.where(nb_ > 0, sb / np.maximum(nb_, 1), cmean)
    clim = shrink_clim(S_, C_, G[CID, bins], k_shrink, smooth)
    return clim, G, edges, bins

def prepare(clim_end):
    ce = MIDX[clim_end]
    mm = np.arange(M) <= ce
    elev = ST[:, STN.index("elev")]
    clim, G, edges, bins = fit_climatology(VI, USABLE, CAL, CID, elev, mm, len(COUNTIES), ELEV_BINS, K_SHRINK, SMOOTH_CLIM)
    anom = np.where(USABLE, VI - clim[:, CAL, :], 0.0).astype("float32")
    an_sd = np.array([anom[:, mm, k][USABLE[:, mm, k]].std() for k in range(K)], "float32")
    anom_z = np.clip(anom / an_sd, -Z_CLIP, Z_CLIP).astype("float32"); del anom
    qual = np.where(CLEAN, 1.0, np.where(USABLE, 0.5, 0.0)).astype("float32")
    wxt = WX.astype("float32").copy(); ri = WXN.index("rain")
    wxt[..., ri] = np.log1p(np.clip(wxt[..., ri], 0, None))
    wx_ok = np.isfinite(wxt).all(-1)
    wx_mu = np.nanmean(wxt[:, mm, :], axis=(0, 1)); wx_sd = np.nanstd(wxt[:, mm, :], axis=(0, 1)) + 1e-6
    wxz = np.nan_to_num((wxt - wx_mu) / wx_sd).astype("float32")
    U4 = np.repeat(wx_ok[:, :, None], 4, 2)
    wclim, wG, _, _ = fit_climatology(np.nan_to_num(wxt), U4, CAL, CID, elev, mm, len(COUNTIES), ELEV_BINS, K_SHRINK, False)
    wanom = np.where(U4, wxt - wclim[:, CAL, :], 0.0)
    wxa_sd = wanom[:, mm, :][wx_ok[:, mm]].std(axis=0) + 1e-6
    wxa = (wanom / wxa_sd).astype("float32"); del wanom
    wxcz = ((wclim - wx_mu) / wx_sd).astype("float32")
    st_mu, st_sd = ST.mean(0), ST.std(0) + 1e-6
    st_full = np.concatenate([(ST - st_mu) / st_sd, np.eye(len(COUNTIES))[CID]], 1).astype("float32")
    menc = np.stack([np.sin(2 * np.pi * CAL / 12), np.cos(2 * np.pi * CAL / 12)], 1).astype("float32")
    return dict(clim_end=clim_end, mm=mm, clim=clim, wG=wG, anom_z=anom_z, qual=qual, wxz=wxz, wxa=wxa, wxcz=wxcz, wx_ok=wx_ok, st_full=st_full, menc=menc,
                an_sd=an_sd, wx_mu=wx_mu, wx_sd=wx_sd, wxa_sd=wxa_sd, st_mu=st_mu, st_sd=st_sd, G=G, edges=edges)

t0 = time.time()
P = prepare(cfg["clim_end"])
print(f"prepared in {time.time()-t0:.0f}s")
for k in ("an_sd", "wx_mu", "wx_sd", "wxa_sd", "st_mu", "st_sd", "G"):
    a, b = np.asarray(P[k], "float64"), np.asarray(norm[k], "float64")
    rel = np.abs(a - b).max() / (np.abs(b).max() + 1e-9)
    print(f"  {k:7s} max relative difference vs saved: {rel:.2e}" + ("   <-- differs: PIX_PER_COUNTY or the dataset is not the one used in B" if rel > 1e-3 else ""))
if "Gw" in norm:
    a, b = np.asarray(P["wG"], "float64"), np.asarray(norm["Gw"], "float64")
    rel = np.abs(a - b).max() / (np.abs(b).max() + 1e-9)
    print(f"  {'Gw':7s} max relative difference vs saved: {rel:.2e}" + ("   <-- differs: PIX_PER_COUNTY or the dataset is not the one used in B" if rel > 1e-3 else ""))
else: print("  Gw      not in the saved norm: weather group means not checked")

## 3. Stores and coarse weather
Opens `manifest.json` and the coarse weather, and uses the same weather chain as A2 (`chain_at`).

In [ ]:
MAN = json.load(open(MANIFEST))
TILE, I16_NODATA, NODATA = int(MAN["tile"]), int(MAN["i16_nodata"]), -9999.0
if MAN["months"] != MONTHS:
    raise RuntimeError(f"the stores cover {MAN['months'][0]}..{MAN['months'][-1]} but the model was trained on {MONTHS[0]}..{MONTHS[-1]}: run Notebook A and B on the same END_MONTH")
FC_LABELS = [str(m) for m in MAN["forecast_months"]]
if len(FC_LABELS) != H or FC_LABELS[0] != str(pd.Period(MONTHS[-1], "M") + 1):
    raise RuntimeError(f"forecast months in the manifest ({FC_LABELS}) do not follow the last observed month {MONTHS[-1]} for H = {H}")
CAL_FUT = np.array([int(m[5:7]) - 1 for m in FC_LABELS])
print("forecast months:", FC_LABELS)
if WXN != ["temp", "rain", "rh", "wind"]: raise RuntimeError(f"chain_at returns temp, rain, rh, wind in this order; the model has {WXN}")
no_store = [v for c in COUNTIES for v in VIS if v not in MAN["counties"][c]["indices"]]
if no_store: raise RuntimeError(f"indices of the model that are not in the stores: {sorted(set(no_store))}")
LAPSE_T, LAPSE_TD = MAN["lapse"]["temp"], MAN["lapse"]["dewpoint"]

A_MAGNUS, B_MAGNUS = 17.625, 243.04
def rh_from_td(t, td):
    return np.clip(100 * np.exp(A_MAGNUS * td / (B_MAGNUS + td)) / np.exp(A_MAGNUS * t / (B_MAGNUS + t)), 0, 100)
def td_from_rh(t, rh):
    g = np.log(np.clip(rh, 1, 100) / 100) + A_MAGNUS * t / (B_MAGNUS + t)
    return B_MAGNUS * g / (A_MAGNUS - g)
def to_crs(xs, ys, src_crs, dst_crs):
    if rasterio.crs.CRS.from_user_input(src_crs) == rasterio.crs.CRS.from_user_input(dst_crs):
        return np.asarray(xs), np.asarray(ys)
    x2, y2 = warp_transform(src_crs, dst_crs, np.asarray(xs).tolist(), np.asarray(ys).tolist())
    return np.asarray(x2), np.asarray(y2)
class Pts:
    def __init__(self, xs, ys, crs):
        self.xs, self.ys, self.crs, self._c = np.asarray(xs), np.asarray(ys), crs, {}
    def in_crs(self, crs):
        k = str(crs)
        if k not in self._c: self._c[k] = to_crs(self.xs, self.ys, self.crs, crs)
        return self._c[k]
def sample_bilinear(arr, tf, xs, ys):
    cf, rf = (~tf) * (xs, ys)
    c, r = cf - 0.5, rf - 0.5
    h, w = arr.shape
    c, r = np.clip(c, 0, w - 1), np.clip(r, 0, h - 1)
    c0, r0 = np.floor(c).astype(int), np.floor(r).astype(int)
    c1, r1 = np.minimum(c0 + 1, w - 1), np.minimum(r0 + 1, h - 1)
    fc, fr = c - c0, r - r0
    top = arr[r0, c0] * (1 - fc) + arr[r0, c1] * fc
    bot = arr[r1, c0] * (1 - fc) + arr[r1, c1] * fc
    return (top * (1 - fr) + bot * fr).astype("float32")
def sample_tif_nearest(path, pts, band=1):
    with rasterio.open(path) as s:
        x2, y2 = pts.in_crs(s.crs)
        cols, rows = (~s.transform) * (x2, y2)
        cols, rows = np.floor(cols).astype(int), np.floor(rows).astype(int)
        inside = (rows >= 0) & (rows < s.height) & (cols >= 0) & (cols < s.width)
        out = np.full(len(x2), np.nan, "float32")
        if not inside.any(): return out
        r0, r1, c0, c1 = rows[inside].min(), rows[inside].max() + 1, cols[inside].min(), cols[inside].max() + 1
        a = s.read(band, window=Window(c0, r0, c1 - c0, r1 - r0), masked=True).astype("float32").filled(np.nan)
        out[inside] = a[rows[inside] - r0, cols[inside] - c0]
    return out
class Coarse:
    def __init__(self, path):
        z = np.load(path); self.arr, self.tf, self.crs = {}, {}, {}
        for v in WXN:
            self.arr[v] = z[f"arr_{v}"]; self.tf[v] = Affine(*z[f"tf_{v}"]); self.crs[v] = rasterio.crs.CRS.from_wkt(str(z[f"crs_{v}"]))
        n = len(self.arr[WXN[0]])
        self.valid = np.array([all(np.isfinite(self.arr[v][i]).any() for v in WXN) for i in range(n)])
def _bilinear_index(shape, tf, xs, ys):
    cf, rf = (~tf) * (xs, ys)
    c, r = cf - 0.5, rf - 0.5
    h, w = shape
    c, r = np.clip(c, 0, w - 1), np.clip(r, 0, h - 1)
    c0, r0 = np.floor(c).astype(int), np.floor(r).astype(int)
    c1, r1 = np.minimum(c0 + 1, w - 1), np.minimum(r0 + 1, h - 1)
    return r0, r1, c0, c1, r - r0, c - c0
def chain_at(C, pts, dz, idx):
    idx = list(idx); out = np.full((len(pts.xs), len(idx), 4), np.nan, "float32")
    dz = np.nan_to_num(np.asarray(dz, "float32"))
    use = [j for j, i in enumerate(idx) if C.valid[i]]
    if not use: return out
    ii = np.array([idx[j] for j in use])[:, None]
    cache = pts.__dict__.setdefault("_bw", {})
    got = {}
    for v in WXN:
        key = (id(C), v)
        if key not in cache:
            x, y = pts.in_crs(C.crs[v]); cache[key] = _bilinear_index(C.arr[v].shape[1:], C.tf[v], x, y)
        r0, r1, c0, c1, fr, fc = cache[key]; A = C.arr[v]
        top = A[ii, r0[None, :], c0[None, :]] * (1 - fc) + A[ii, r0[None, :], c1[None, :]] * fc
        bot = A[ii, r1[None, :], c0[None, :]] * (1 - fc) + A[ii, r1[None, :], c1[None, :]] * fc
        got[v] = (top * (1 - fr) + bot * fr).astype("float32")         # [len(use), n]
    Tm, RH = got["temp"], got["rh"]
    Td = td_from_rh(Tm, RH); Tc, Tdc = Tm + LAPSE_T * dz[None, :], Td + LAPSE_TD * dz[None, :]
    res = (Tc, got["rain"], rh_from_td(Tc, Tdc), got["wind"])
    for k in range(4): out[:, use, k] = res[k].T
    return out

TPL = {}
for c in COUNTIES:
    m = MAN["counties"][c]
    TPL[c] = dict(crs=rasterio.crs.CRS.from_wkt(m["crs"]), transform=Affine(*m["transform"]), shape=tuple(m["shape"]), ny=m["ny"], nx=m["nx"], nt=m["n_tiles"],
                  static=m["static"], dz=m["dz"], store_dir=m["store_dir"], indices=m["indices"])
    print(f"{c}: grid {TPL[c]['shape'][0]} x {TPL[c]['shape'][1]} ({np.prod(TPL[c]['shape']) / 1e6:.1f} M pixels), {TPL[c]['nt']} tiles of {TILE} x {TILE}")
COARSE = {}
for c in COUNTIES:
    COARSE[(c, "hist")] = Coarse(MAN["counties"][c]["coarse_hist"])
    bad = [MONTHS[i] for i in HIST if not COARSE[(c, "hist")].valid[i]]
    if bad: raise RuntimeError(f"{c}: the coarse historical weather has no complete data for {bad}; fix it in Notebook A")
    for s in MAN["scenarios"]:
        if s in MAN["counties"][c]["coarse_forecast"]: COARSE[(c, s)] = Coarse(MAN["counties"][c]["coarse_forecast"][s])
ACTIVE = []
for s in MAN["scenarios"]:
    if not all((c, s) in COARSE for c in COUNTIES): print(f"!! scenario {s} is not available for every county: skipped"); continue
    gaps = {c: [FC_LABELS[i] for i in range(H) if not COARSE[(c, s)].valid[i]] for c in COUNTIES}
    if any(gaps.values()): print(f"!! scenario {s} has forecast months without complete weather {gaps}: skipped"); continue
    ACTIVE.append(s)
if CENTRAL not in ACTIVE: raise RuntimeError(f"the central scenario '{CENTRAL}' is not available; scenarios usable: {ACTIVE}")
bad_full = [s for s in FULL_SCENARIOS if s not in ACTIVE + ["normal"]]
if bad_full: raise RuntimeError(f"FULL_SCENARIOS contains {bad_full}, which are not available; usable: {ACTIVE + ['normal']}")
print("forecast scenarios used:", ACTIVE)

FC_RAW = {s: {w: np.full((N, H), np.nan, "float32") for w in WXN} for s in ACTIVE}
for ci, c in enumerate(COUNTIES):
    idx = np.flatnonzero(CID == ci)
    pts = Pts(COORDS[0][idx], COORDS[1][idx], TPL[c]["crs"])
    for s in ACTIVE:
        f = chain_at(COARSE[(c, s)], pts, DZ[idx], list(range(H)))
        for k, w in enumerate(WXN): FC_RAW[s][w][idx] = f[:, :, k]
print("forecast weather at the sample pixels:", {s: f"{np.isfinite(FC_RAW[s]['temp']).all(1).mean():.1%} complete" for s in ACTIVE})

## 4. Optional extra bias correction and weather check
`BIAS_CSV = None`: SEAS5 is already bias-corrected upstream. The table compares the forecast with the training weather of the same month.

In [ ]:
START_MONTH_FC = int(FC_LABELS[0][5:7])
def load_bias_table():
    if BIAS_CSV is None or not os.path.exists(BIAS_CSV):
        print("no extra bias table: forecast weather is already corrected upstream (BC_V2)" if BIAS_CSV is None else f"!! no bias table at {BIAS_CSV} -> NO extra bias correction applied"); return None
    b = pd.read_csv(BIAS_CSV)
    need = {"county", "var", "start_month", "lead", "value", "kind"}
    if not need.issubset(b.columns): raise RuntimeError(f"{BIAS_CSV} must have the columns {sorted(need)} (the file written by Notebook D); found {list(b.columns)}")
    b = b[(b.start_month == START_MONTH_FC) & (b.lead.between(0, H - 1))].copy()
    print(f"bias table from {BIAS_CSV}: start month {START_MONTH_FC}, {b.value.notna().sum()} corrected cells of {len(b)}")
    return b
BIAS = load_bias_table()
BIAS_LUT = {} if BIAS is None else {(r.county, r.var, int(r.lead)): (r.kind, float(r.value)) for r in BIAS.itertuples() if np.isfinite(r.value)}
print("variables corrected per county:", {c: sorted({v for (cc, v, _) in BIAS_LUT if cc == c}) for c in COUNTIES})

def correct(name, vals, county):
    out = np.array(vals, "float32", copy=True)
    for j in range(H):
        e = BIAS_LUT.get((county, name, j))
        if e is None: continue
        out[:, j] = out[:, j] - e[1] if e[0] == "delta" else out[:, j] * e[1]
    if name == "rh": out = np.clip(out, 0, 100)
    elif name in ("rain", "wind"): out = np.clip(out, 0, None)
    return out
def correct_all(name, vals, cid):
    out = np.array(vals, "float32", copy=True)
    for ci, c in enumerate(COUNTIES):
        m = cid == ci
        if m.any(): out[m] = correct(name, vals[m], c)
    return out

FC = {s: {w: correct_all(w, FC_RAW[s][w], CID) for w in WXN} for s in ACTIVE}

rows_ = []
for ci, county in enumerate(COUNTIES):
    pc = CID == ci
    for w in WXN:
        wi = WXN.index(w)
        for j in range(H):
            hist = np.nanmean(WX[pc][:, CAL == CAL_FUT[j], wi])
            raw, cor = np.nanmean(FC_RAW[CENTRAL][w][pc, j]), np.nanmean(FC[CENTRAL][w][pc, j])
            rows_.append(dict(county=county, var=w, month=FC_LABELS[j], training_mean=hist, forecast_raw=raw, forecast_corrected=cor, ratio_corrected=cor / hist if hist else np.nan, diff_corrected=cor - hist))
SANITY = pd.DataFrame(rows_)
print("\ncorrected central forecast vs training weather of the same calendar month (county means):")
print(SANITY.round(2).to_string(index=False))
bad = SANITY[((SANITY["var"] == "rain") & ((SANITY["ratio_corrected"] < 0.3) | (SANITY["ratio_corrected"] > 3))) |
             ((SANITY["var"] == "temp") & (SANITY["diff_corrected"].abs() > 3)) | ((SANITY["var"] == "rh") & (SANITY["diff_corrected"].abs() > 15))]
if len(bad): print("\n!! these rows are far from the training weather (check units, e.g. rain per day vs per month, or the bias correction):"); print(bad.round(2).to_string(index=False))

## 5. Decoder inputs for each scenario
Weather anomalies from each pixel's own climatology, damped by `RECENTRE`, `SHRINK` and `ANOM_CAP`. The `normal` scenario has zero anomaly.

In [ ]:
ri = WXN.index("rain")
wclim = P["wxcz"] * P["wx_sd"] + P["wx_mu"]
MENC_FUT = np.stack([np.sin(2 * np.pi * CAL_FUT / 12), np.cos(2 * np.pi * CAL_FUT / 12)], 1).astype("float32")      # [H, 2]
ST_IDX = [MAN["static_names"].index(s) for s in STN]
ELEV_COL = STN.index("elev")
I_QA = VIS.index(QA_SOURCE) if (QA_SOURCE and QA_SOURCE in VIS) else None
INHERIT = I_QA is not None and bool(has_qa[I_QA])

def points_weather(c, ci, pts, dz, bins, scens):
    n = len(pts.xs)
    ch = chain_at(COARSE[(c, "hist")], pts, dz, list(range(M)))                    # [n, M, 4] raw units
    ch[..., ri] = np.log1p(np.clip(ch[..., ri], 0, None))
    ok4 = np.isfinite(ch).all(-1)                                                  # [n, M]
    chz = np.nan_to_num(ch)
    S_ = np.zeros((n, 12, 4), "float32"); C_ = np.zeros((n, 12, 4), "float32")
    for cm in range(12):
        sel = np.flatnonzero((CAL == cm) & P["mm"])
        if len(sel) == 0: continue
        u = ok4[:, sel][:, :, None]
        S_[:, cm, :] = np.where(u, chz[:, sel, :], 0).sum(1)
        C_[:, cm, :] = u.sum(1)
    wcl = shrink_clim(S_, C_, P["wG"][ci][bins], K_SHRINK, False)
    h_ok = ok4[:, HIST]
    wxa_h = (np.where(h_ok[:, :, None], chz[:, HIST, :] - wcl[:, CAL[HIST], :], 0.0) / P["wxa_sd"]).astype("float32")
    wxz_h = np.nan_to_num((ch[:, HIST, :] - P["wx_mu"]) / P["wx_sd"]).astype("float32")
    fc = {}
    for s in scens:
        if s == "normal": continue
        f = chain_at(COARSE[(c, s)], pts, dz, list(range(H)))
        for k, w in enumerate(WXN): f[:, :, k] = correct(w, f[:, :, k], c)
        fc[s] = f
    return dict(wclim=wcl, wxz_h=wxz_h, wxa_h=wxa_h, wx_hist_ok=h_ok.all(1), fc=fc)

def build_inputs(V, QA_, ci, bins, stat_sel, hw):
    n = len(V)
    U = np.isfinite(V) & ((QA_ == 0) | (QA_ == 2))
    S_ = np.zeros((n, 12, K), "float32"); C_ = np.zeros((n, 12, K), "float32")
    for cm in range(12):
        sel = np.flatnonzero((CAL == cm) & P["mm"])
        if len(sel) == 0: continue
        u = U[:, sel, :]
        S_[:, cm, :] = np.where(u, V[:, sel, :], 0).sum(1)
        C_[:, cm, :] = u.sum(1)
    clim = shrink_clim(S_, C_, P["G"][ci][bins], K_SHRINK, SMOOTH_CLIM)            # [n, 12, K]
    Vh, Uh, Qh = V[:, HIST, :], U[:, HIST, :], QA_[:, HIST, :].copy()
    if INHERIT:
        gf = Qh[:, :, I_QA] == 2
        for k in range(K):
            if not has_qa[k]: Qh[:, :, k] = np.where(gf & np.isfinite(Vh[:, :, k]) & (Qh[:, :, k] == 0), 2, Qh[:, :, k])
    clean = np.isfinite(Vh) & (Qh == 0)
    qual = np.where(clean, 1.0, np.where(Uh, 0.5, 0.0))
    anom = np.where(Uh, Vh - clim[:, CAL[HIST], :], 0.0)
    anom_z = np.clip(anom / P["an_sd"], -Z_CLIP, Z_CLIP)
    n_prim = Uh[:, :, PRI_K].sum(1)
    enc = [anom_z, qual] + ([hw["wxz_h"]] if WX_INPUT == "both" else []) + [hw["wxa_h"], np.broadcast_to(P["menc"][HIST], (n, L, 2))]
    XE_ = np.concatenate(enc, -1).astype("float32")
    stz = np.concatenate([(stat_sel - P["st_mu"]) / P["st_sd"], np.broadcast_to(np.eye(len(COUNTIES))[ci], (n, len(COUNTIES)))], 1).astype("float32")
    return XE_, stz, clim, hw["wx_hist_ok"] & (n_prim >= MIN_ENC_USABLE)

def raw_anomaly(wx_fc, wclim_fut):
    wxt = np.asarray(wx_fc, "float32").copy()
    wxt[..., ri] = np.log1p(np.clip(wxt[..., ri], 0, None))
    return (wxt - wclim_fut) / P["wxa_sd"]

def decoder_core(wx_fc, wclim_fut, cid, scen, off):
    n = len(wclim_fut)
    if scen == "normal":
        raw = (wclim_fut - P["wx_mu"]) / P["wx_sd"]; an = np.zeros_like(raw); ok = np.ones(n, bool)
    else:
        ok = np.isfinite(np.asarray(wx_fc)).all(axis=(1, 2))
        an = raw_anomaly(wx_fc, wclim_fut)                                  # NaN where the forecast is missing
        for k, w in enumerate(WXN):
            if RECENTRE[w]: an[..., k] -= RECENTRE[w] * off[cid, k][:, None]
            an[..., k] = np.clip(SHRINK[w] * an[..., k], -ANOM_CAP, ANOM_CAP)
        raw = (wclim_fut - P["wx_mu"]) / P["wx_sd"] + an * P["wxa_sd"] / P["wx_sd"]
        raw, an = np.nan_to_num(raw).astype("float32"), np.nan_to_num(an).astype("float32")
    parts = ([raw] if WX_INPUT == "both" else []) + [an, np.broadcast_to(MENC_FUT, (n, H, 2))]
    return np.concatenate(parts, -1).astype("float32"), ok

def draw_random_pixels(c, n, seed):
    with rasterio.open(TPL[c]["static"]) as s: elev = s.read(1)
    ok = np.flatnonzero(np.isfinite(elev).ravel())
    pick = np.random.default_rng(seed).choice(ok, min(n, len(ok)), replace=False)
    rows, cols = np.divmod(pick, elev.shape[1])
    return rows, cols, elev[rows, cols]

AN_OFF = {s: np.zeros((len(COUNTIES), 4)) for s in ACTIVE}                    # scenario -> [county, variable]
for ci, c in enumerate(COUNTIES):
    rows, cols, el = draw_random_pixels(c, OFFSET_PIXELS, 7 + ci)
    xs, ys = TPL[c]["transform"] * (cols + 0.5, rows + 0.5)
    pts = Pts(xs, ys, TPL[c]["crs"]); dz = sample_tif_nearest(TPL[c]["dz"], pts)
    hw = points_weather(c, ci, pts, dz, np.digitize(el, P["edges"][ci]), ACTIVE)
    for s in ACTIVE:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            AN_OFF[s][ci] = np.nanmean(raw_anomaly(hw["fc"][s], hw["wclim"][:, CAL_FUT, :]), axis=(0, 1))
print(f"county-mean forecast weather anomaly over the {H} months, in standardised units (RECENTRE removes {RECENTRE} of it):")
print(pd.concat({s: pd.DataFrame(AN_OFF[s], index=COUNTIES, columns=WXN) for s in ACTIVE}).round(2).to_string())

# ---- sample pixels ----
def decoder_inputs(scen):
    wx_fc = None if scen == "normal" else np.stack([FC[scen][w] for w in WXN], axis=2)
    return decoder_core(wx_fc, wclim[:, CAL_FUT, :], CID, scen, AN_OFF.get(scen))

enc = [P["anom_z"][:, HIST, :], P["qual"][:, HIST, :]] + ([P["wxz"][:, HIST, :]] if WX_INPUT == "both" else []) + [P["wxa"][:, HIST, :], np.broadcast_to(P["menc"][HIST], (N, L, 2))]
XE = np.concatenate(enc, -1).astype("float32")
STZ = P["st_full"]
n_prim = USABLE[:, HIST, PRI_K].sum(1)
ENC_OK = wx_hist_ok & P["wx_ok"][:, HIST].all(1) & (n_prim >= MIN_ENC_USABLE)
print(f"pixels with a valid encoder window (complete weather, >= {MIN_ENC_USABLE} usable {PRIMARY} months): {ENC_OK.mean():.1%}")
print("pixels per county:", {c: int(((CID == i) & ENC_OK).sum()) for i, c in enumerate(COUNTIES)})

## 6. Run the ensemble

In [ ]:
class Seq2SeqLSTM(nn.Module):
    def __init__(self, n_enc, n_dec, n_static, n_out, hidden, layers, dropout, s_emb):
        super().__init__()
        self.static = nn.Sequential(nn.Linear(n_static, s_emb), nn.ReLU())
        p = dropout if layers > 1 else 0.0
        self.enc = nn.LSTM(n_enc + s_emb, hidden, layers, batch_first=True, dropout=p)
        self.dec = nn.LSTM(n_dec + s_emb, hidden, layers, batch_first=True, dropout=p)
        self.drop, self.head = nn.Dropout(dropout), nn.Linear(hidden, n_out)
    def encode(self, xe, st):
        s = self.static(st)
        _, state = self.enc(torch.cat([xe, s[:, None, :].expand(-1, xe.size(1), -1)], -1))
        return state, s
    def decode(self, state, s, xd):
        y, _ = self.dec(torch.cat([xd, s[:, None, :].expand(-1, xd.size(1), -1)], -1), state)
        return self.head(self.drop(y))
    def forward(self, xe, xd, st):
        state, s = self.encode(xe, st)
        return self.decode(state, s, xd)

MODELS = []
for sd_ in art["states"][:SEEDS_USED]:
    m_ = Seq2SeqLSTM(2 * K + N_W + 2, N_W + 2, cfg["n_static"], K, cfg["HIDDEN"], cfg["LAYERS"], cfg["DROPOUT"], cfg["STATIC_EMB"]).to(DEVICE)
    m_.load_state_dict(sd_); m_.eval(); MODELS.append(m_)
print(len(MODELS), "models loaded | encoder channels", XE.shape[-1], "| decoder channels", N_W + 2)

@torch.inference_mode()
def predict(xe, xd, st, chunk=None):
    chunk = chunk or PREDICT_CHUNK; outs = []
    for s in range(0, len(xe), chunk):
        a = torch.as_tensor(xe[s:s + chunk], device=DEVICE); b = torch.as_tensor(xd[s:s + chunk], device=DEVICE); c = torch.as_tensor(st[s:s + chunk], device=DEVICE)
        outs.append(torch.stack([m_(a, b, c) for m_ in MODELS]).cpu().numpy())
    return np.concatenate(outs, 1)                                  # [n_models, N, H, K] standardised anomalies

@torch.inference_mode()
def predict_multi(xe, xds, st, chunk=None):
    chunk = chunk or PREDICT_CHUNK; acc = {s: [] for s in xds}
    for s0 in range(0, len(xe), chunk):
        a = torch.as_tensor(xe[s0:s0 + chunk], device=DEVICE); c = torch.as_tensor(st[s0:s0 + chunk], device=DEVICE)
        enc = [m_.encode(a, c) for m_ in MODELS]
        for scen, xd in xds.items():
            b = torch.as_tensor(xd[s0:s0 + chunk], device=DEVICE)
            acc[scen].append(torch.stack([m_.decode(st_, s_, b) for m_, (st_, s_) in zip(MODELS, enc)]).mean(0).cpu().numpy())
    return {s: np.concatenate(v, 0) for s, v in acc.items()}

SCEN_ALL = ACTIVE + ["normal"]
CLIM_FUT = P["clim"][:, CAL_FUT, :]                                 # [N, H, K]
VALUE  = np.full((len(SCEN_ALL), N, H, K), np.nan, "float32")       # forecast index values
ANOM   = np.full_like(VALUE, np.nan)                                # forecast anomalies in index units
SPREAD = np.full_like(VALUE, np.nan)                                # sd across the ensemble members, index units
n_clipped = 0
t0 = time.time()
for si, scen in enumerate(SCEN_ALL):
    xd, wok = decoder_inputs(scen)
    ok = ENC_OK & wok
    pred = predict(np.nan_to_num(XE), xd, STZ)                      # [S, N, H, K]
    mean, sd = pred.mean(0), pred.std(0)
    an = mean * P["an_sd"]
    val = CLIM_FUT + an
    n_clipped += int(((val < VAL_LO) | (val > VAL_HI))[ok].sum())
    val = np.clip(val, VAL_LO, VAL_HI)
    VALUE[si][ok], ANOM[si][ok], SPREAD[si][ok] = val[ok], an[ok], (sd * P["an_sd"])[ok]
    print(f"  scenario {scen:6s}: {ok.sum()} pixels, {time.time()-t0:.0f}s", flush=True)
print(f"values clipped to the plausible range: {n_clipped} ({n_clipped / max(np.isfinite(VALUE).sum(), 1):.3%})")
SI = {s: i for i, s in enumerate(SCEN_ALL)}

## 7. Save sample-pixel results

In [ ]:
np.savez_compressed(os.path.join(OUT_DIR, "forecast_pixels.npz"),
    value=VALUE, anomaly=ANOM, model_spread=SPREAD, climatology=CLIM_FUT.astype("float32"),
    scenarios=np.array(SCEN_ALL), vis=np.array(VIS), months=np.array(FC_LABELS),
    county_id=CID, counties=np.array(COUNTIES), pixel_index_in_npz=keep, valid=ENC_OK,
    x=(COORDS[0] if COORDS is not None else np.full(N, np.nan)), y=(COORDS[1] if COORDS is not None else np.full(N, np.nan)))

skill = pd.read_csv(SKILL_CSV, index_col=0) if os.path.exists(SKILL_CSV) else None      # rows "lead 1".."lead 7", columns = indices
rows_ = []
for scen in SCEN_ALL:
    si = SI[scen]
    for ci, county in enumerate(COUNTIES):
        pm = (CID == ci) & ENC_OK
        if pm.sum() == 0: continue
        for k, v in enumerate(VIS):
            for h in range(H):
                x = VALUE[si][pm][:, h, k]
                rows_.append(dict(scenario=scen, county=county, index=v, month=FC_LABELS[h], lead=h + 1, n_pixels=int(pm.sum()),
                                  value_mean=np.nanmean(x), value_p10_pixels=np.nanpercentile(x, 10), value_p90_pixels=np.nanpercentile(x, 90),
                                  climatology_mean=np.nanmean(CLIM_FUT[pm][:, h, k]), anomaly_mean=np.nanmean(ANOM[si][pm][:, h, k]),
                                  model_spread_mean=np.nanmean(SPREAD[si][pm][:, h, k]),
                                  test_skill_B=(skill.loc[f"lead {h+1}", v] if skill is not None and v in skill.columns else np.nan)))
SUMMARY = pd.DataFrame(rows_)
SUMMARY.to_csv(os.path.join(OUT_DIR, "forecast_county_summary.csv"), index=False)
SANITY.to_csv(os.path.join(OUT_DIR, "forecast_weather_check.csv"), index=False)
print("saved to", OUT_DIR, ":", ["forecast_pixels.npz", "forecast_county_summary.csv", "forecast_weather_check.csv"])

cm = SUMMARY[SUMMARY["scenario"] == CENTRAL]
print(f"\nPRIMARY index ({PRIMARY}), {CENTRAL} scenario: county mean forecast value (anomaly in brackets)")
tab = cm[cm["index"] == PRIMARY].assign(txt=lambda d: d["value_mean"].round(3).astype(str) + " (" + d["anomaly_mean"].round(3).astype(str) + ")").pivot(index="county", columns="month", values="txt")
print(tab.to_string())

## 8. Plots and reliability table
Sample pixels: observed history, central forecast, p10 to p90 band and the `normal` line.

In [ ]:
OBS = np.where(USABLE, VI, np.nan)
nobs = 18
fig, axes = plt.subplots(len(PLOT_INDICES), len(COUNTIES), figsize=(5 * len(COUNTIES), 3.1 * len(PLOT_INDICES)), squeeze=False)
xo = np.arange(-nobs + 1, 1); xf = np.arange(1, H + 1)
for r, v in enumerate(PLOT_INDICES):
    k = VIS.index(v)
    for c, county in enumerate(COUNTIES):
        ax = axes[r, c]; pm = (CID == c) & ENC_OK
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            ax.plot(xo, np.nanmean(OBS[pm][:, M - nobs:, k], 0), "k.-", label="observed (sample mean)")
            cen = np.nanmean(VALUE[SI[CENTRAL]][pm][:, :, k], 0)
            ax.plot(xf, cen, "o-", color="tab:blue", label=f"forecast ({CENTRAL})")
            if "p10" in SI and "p90" in SI:
                lo, hi = np.nanmean(VALUE[SI["p10"]][pm][:, :, k], 0), np.nanmean(VALUE[SI["p90"]][pm][:, :, k], 0)
                ax.fill_between(xf, np.minimum(lo, hi), np.maximum(lo, hi), color="tab:blue", alpha=.2, label="p10-p90 weather scenarios")
            ax.plot(xf, np.nanmean(VALUE[SI["normal"]][pm][:, :, k], 0), "--", color="tab:gray", label="normal weather")
        ax.axvline(0.5, color="k", lw=.5); ax.set_title(f"{county}: {v}"); ax.set_xlabel("months from forecast origin (" + MONTHS[-1] + ")")
        if r == 0 and c == 0: ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "forecast_county_plots.png"), dpi=140); plt.show()

if skill is not None:
    print("test skill vs climatology from Notebook B (observed weather = upper bound; SEAS5 weather will be lower):")
    print(skill.round(2).to_string())
    weak = [(v, f"lead {h+1}") for v in VIS if v in skill.columns for h in range(H) if skill.loc[f"lead {h+1}", v] <= 0]
    print("\nindex/lead pairs with test skill <= 0 (treat the forecast as 'normal month' only):", weak if weak else "none")
else:
    print("test-skill table not found at", SKILL_CSV)

if COORDS is not None:
    k = PRI_K
    fig, axes = plt.subplots(len(COUNTIES), 2, figsize=(9, 3.6 * len(COUNTIES)), squeeze=False)
    for ci, county in enumerate(COUNTIES):
        pm = (CID == ci) & ENC_OK
        for j, h in enumerate((0, H - 1)):
            sc_ = axes[ci, j].scatter(COORDS[0][pm], COORDS[1][pm], c=ANOM[SI[CENTRAL]][pm][:, h, k], s=2, cmap="RdYlGn", vmin=-0.2, vmax=0.2)
            axes[ci, j].set_title(f"{county}: {PRIMARY} anomaly, {FC_LABELS[h]} (lead {h+1})"); axes[ci, j].set_aspect("equal"); plt.colorbar(sc_, ax=axes[ci, j])
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "forecast_anomaly_maps.png"), dpi=140); plt.show()

## Faster full-area run
- Use a GPU if available, or `SEEDS_USED = 1` for a draft.
- Time a few tiles first: `MAX_TILES_PER_COUNTY = 5`, `COUNTIES_TO_RUN = ["Nyeri"]`.
- Keep `FULL_SCENARIOS = ["mean"]`.
- The run is resumable: finished tiles stay in `tile_cache`.
- `DELETE_TILE_CACHE = True` frees disk after the GeoTIFFs are written.

## 9. Full-area forecast, tile by tile
For each 256 x 256 tile: read the stores, build the weather inputs, run the ensemble and convert anomalies back to index values. Results go to `tile_cache`; interrupted runs resume.

**Self-check first:** the tile code is run on sample pixels and must agree with sections 1-8 (differences around 1e-4 or less). If not, do not trust the maps.

In [ ]:
COUNTIES_TO_RUN = ["Kakamega", "Nyeri", "Kilifi"]     # e.g. ["Nyeri"] to test on one county first

def check_tile_pipeline(n_per_county=2000):
    rng = np.random.default_rng(3); rows = []
    for ci, c in enumerate(COUNTIES):
        idx = np.sort(rng.choice(np.flatnonzero(CID == ci), min(n_per_county, int((CID == ci).sum())), replace=False))
        pts = Pts(COORDS[0][idx], COORDS[1][idx], TPL[c]["crs"])
        bins = np.digitize(ST[idx, ELEV_COL], P["edges"][ci])
        hw = points_weather(c, ci, pts, DZ[idx], bins, [])
        XE_, STZ_, clim, ok = build_inputs(VI[idx], QA[idx], ci, bins, ST[idx], hw)
        sl = {"anomalies": slice(0, K), "quality": slice(K, 2 * K), "weather anomalies": slice(XE.shape[-1] - 6, XE.shape[-1] - 2)}
        d = {n_: float(np.abs(XE_[..., s_] - XE[idx][..., s_]).max()) for n_, s_ in sl.items()}
        rows.append(dict(county=c, n=len(idx), **{f"max diff {k_}": v_ for k_, v_ in d.items()},
                         **{"max diff climatology": float(np.abs(clim - P["clim"][idx]).max()), "max diff static": float(np.abs(STZ_ - P["st_full"][idx]).max()),
                            "same valid-window flag": float((ok == ENC_OK[idx]).mean())}))
    tab = pd.DataFrame(rows); print(tab.round(6).to_string(index=False))
    worst = tab.filter(like="max diff").to_numpy().max()
    print("\nOK: the tile pipeline reproduces the sample inputs." if worst < 1e-2 and (tab["same valid-window flag"] > 0.999).all()
          else "\n!! the tile pipeline does NOT reproduce the sample inputs: check the stores, dz and the coarse weather before using the maps")
check_tile_pipeline()

# ---------- stores, tile cache ----------
def open_stores(c):
    tpl, d = TPL[c], {}
    for v in VIS:
        info = tpl["indices"][v]; shp = (M, tpl["nt"], TILE, TILE)
        qa = np.memmap(os.path.join(tpl["store_dir"], f"qa_{v}.bin"), np.uint8, "r", shape=shp) if info["has_qa"] else None
        d[v] = (np.memmap(os.path.join(tpl["store_dir"], f"{v}.bin"), np.int16, "r", shape=shp), float(info["scale"]), qa)
    return d

def tile_ids(c):
    with rasterio.open(TPL[c]["static"]) as s: elev = s.read(1)
    if elev.shape != TPL[c]["shape"]: raise RuntimeError(f"{c}: the static layer {elev.shape} does not match the grid in the manifest {TPL[c]['shape']}")
    ny, nx = TPL[c]["ny"], TPL[c]["nx"]
    pad = np.zeros((ny * TILE, nx * TILE), bool); pad[:elev.shape[0], :elev.shape[1]] = np.isfinite(elev)
    return np.flatnonzero(pad.reshape(ny, TILE, nx, TILE).any(axis=(1, 3)).ravel())

def prep_tile(c, ci, t, stores, scens):
    tpl = TPL[c]; hgt, wid = tpl["shape"]; ty, tx = divmod(t, tpl["nx"]); r0, c0 = ty * TILE, tx * TILE
    h_, w_ = min(TILE, hgt - r0), min(TILE, wid - c0)
    with rasterio.open(tpl["static"]) as static_ds:
        stat = static_ds.read(window=Window(c0, r0, w_, h_)).astype("float32")             # [5, h, w], NaN outside the county
    inside = np.isfinite(stat).all(0)
    if not inside.any(): return None
    rr, cc = np.nonzero(inside); n = len(rr)
    stat_sel = stat[ST_IDX][:, rr, cc].T
    V = np.empty((n, M, K), "float32"); QA_ = np.empty((n, M, K), "uint8")
    for k, v in enumerate(VIS):
        mv, sc, mq = stores[v]
        raw = np.asarray(mv[:, t])[:, rr, cc].T                                            # [n, M] int16
        V[:, :, k] = np.where(raw == I16_NODATA, np.nan, raw / sc)
        QA_[:, :, k] = np.asarray(mq[:, t])[:, rr, cc].T if mq is not None else np.where(np.isfinite(V[:, :, k]), 0, 255)
    xs, ys = tpl["transform"] * (c0 + cc + 0.5, r0 + rr + 0.5)
    pts = Pts(xs, ys, tpl["crs"]); dz = sample_tif_nearest(tpl["dz"], pts)
    bins = np.digitize(stat_sel[:, ELEV_COL], P["edges"][ci])
    hw = points_weather(c, ci, pts, dz, bins, scens)
    XE_, STZ_, clim, enc_ok = build_inputs(V, QA_, ci, bins, stat_sel, hw)
    XE_ = np.nan_to_num(XE_); cid = np.full(n, ci)
    dec = {scen: decoder_core(hw["fc"].get(scen), hw["wclim"][:, CAL_FUT, :], cid, scen, AN_OFF.get(scen)) for scen in scens}
    return dict(rr=rr, cc=cc, XE=XE_, STZ=STZ_, clim_fut=clim[:, CAL_FUT, :], enc_ok=enc_ok, dec=dec)

def infer_tile(pr, scens):
    if pr is None: return None
    preds = predict_multi(pr["XE"], {s: pr["dec"][s][0] for s in scens}, pr["STZ"])
    out = {}
    for scen in scens:
        val = np.clip(pr["clim_fut"] + preds[scen] * P["an_sd"], VAL_LO, VAL_HI).astype("float32")
        val[~(pr["enc_ok"] & pr["dec"][scen][1])] = np.nan
        out[scen] = (pr["rr"], pr["cc"], val)
    return out

def process_tile(c, ci, t, static_ds, stores, scens):
    return infer_tile(prep_tile(c, ci, t, stores, scens), scens)

def ordered_prefetch(items, fn, window):
    from collections import deque
    from concurrent.futures import ThreadPoolExecutor
    items = iter(items); q = deque()
    with ThreadPoolExecutor(max_workers=max(1, window)) as ex:
        for it in items:
            q.append((it, ex.submit(fn, it)))
            if len(q) >= window + 1: break
        while q:
            it, fut = q.popleft()
            nxt = next(items, None)
            if nxt is not None: q.append((nxt, ex.submit(fn, nxt)))
            yield it, fut.result()

SIG = hashlib.md5(json.dumps(dict(trained=cfg["trained"], shrink=SHRINK, recentre=RECENTRE, cap=ANOM_CAP, clip=CLIP_MARGIN, scens=FULL_SCENARIOS,
                                  off={s: np.round(AN_OFF[s], 5).tolist() for s in AN_OFF}, bias=(None if BIAS is None else BIAS.to_json())), sort_keys=True).encode()).hexdigest()
cache_paths = lambda scen, c: (os.path.join(TILE_CACHE, scen, f"{c}.f32"), os.path.join(TILE_CACHE, scen, f"{c}.json"))
TILE_IDS = {c: tile_ids(c) for c in COUNTIES}

def run_full_area():
    t_start = time.time()
    for ci, c in enumerate(COUNTIES):
        if c not in COUNTIES_TO_RUN: continue
        ids, mms, status = TILE_IDS[c], {}, {}
        for scen in FULL_SCENARIOS:
            path, spath = cache_paths(scen, c); os.makedirs(os.path.dirname(path), exist_ok=True)
            st = json.load(open(spath)) if (os.path.exists(spath) and os.path.exists(path) and not FORCE_RECOMPUTE) else {}
            if st.get("sig") != SIG: st = dict(sig=SIG, done=[])
            mms[scen] = np.memmap(path, "float32", "r+" if (os.path.exists(path) and st["done"]) else "w+", shape=(H, K, len(ids), TILE, TILE)); status[scen] = st
        done = set.intersection(*[set(status[s]["done"]) for s in FULL_SCENARIOS])
        todo = [j for j in range(len(ids)) if j not in done]
        print(f"{c}: {len(ids)} tiles with pixels, {len(done)} already done, {len(todo)} to do", flush=True)
        if not todo: continue
        stores = open_stores(c); t0 = time.time()
        if MAX_TILES_PER_COUNTY: todo = todo[:MAX_TILES_PER_COUNTY]
        prep = lambda j: prep_tile(c, ci, int(ids[j]), stores, FULL_SCENARIOS)
        for n_, (j, pr) in enumerate(ordered_prefetch(todo, prep, PREFETCH_WORKERS), 1):
            res = infer_tile(pr, FULL_SCENARIOS)
            for scen in FULL_SCENARIOS:
                buf = np.full((H, K, TILE, TILE), np.nan, "float32")
                if res is not None:
                    rr, cc, val = res[scen]; buf[:, :, rr, cc] = np.transpose(val, (1, 2, 0))
                mms[scen][:, :, j] = buf; status[scen]["done"].append(int(j))
            if n_ % 10 == 0 or n_ == len(todo):
                for scen in FULL_SCENARIOS:
                    mms[scen].flush(); json.dump(status[scen], open(cache_paths(scen, c)[1], "w"))
                el = time.time() - t0
                print(f"  {c}: {n_}/{len(todo)} tiles, {el / 60:.1f} min, about {el / n_ * (len(todo) - n_) / 60:.0f} min left", flush=True)
        del mms
    print(f"done in {(time.time() - t_start) / 60:.1f} min")

if RUN_FULL_AREA: run_full_area()
else: print("RUN_FULL_AREA = False: full-area run skipped")

## 10. Write the GeoTIFFs
`F:\PREDICTED_INDICES\geotiff\<scenario>\<County>\<County>_<INDEX>_<YYYY-MM>.tif`: float32, no-data -9999, on the stores' 30 m grid. `forecast_geotiff_index.csv` lists every file.

In [ ]:
def from_tiles(tiles, h, w):
    ny, nx = -(-h // TILE), -(-w // TILE)
    return tiles.reshape(ny, nx, TILE, TILE).transpose(0, 2, 1, 3).reshape(ny * TILE, nx * TILE)[:h, :w]

def write_geotiffs():
    from concurrent.futures import ThreadPoolExecutor
    rows_, preview = [], {}
    for scen in FULL_SCENARIOS:
        for ci, c in enumerate(COUNTIES):
            if c not in COUNTIES_TO_RUN: continue
            tpl = TPL[c]; hgt, wid = tpl["shape"]; ids = TILE_IDS[c]; path, spath = cache_paths(scen, c)
            st = json.load(open(spath)) if os.path.exists(spath) else {}
            if st.get("sig") != SIG or len(set(st.get("done", []))) < len(ids):
                print(f"!! {c} / {scen}: the tile cache is not complete ({len(set(st.get('done', [])))}/{len(ids)} tiles): run section 9 again"); continue
            mm = np.memmap(path, "float32", "r", shape=(H, K, len(ids), TILE, TILE))
            d_out = os.path.join(TIFF_DIR, scen, c); os.makedirs(d_out, exist_ok=True)
            prof = dict(driver="GTiff", height=hgt, width=wid, count=1, dtype="float32", crs=tpl["crs"], transform=tpl["transform"], nodata=NODATA,
                        compress="deflate", predictor=3, tiled=True, blockxsize=256, blockysize=256)
            def write_one(kh):
                k, h = kh; v = VIS[k]
                full = np.full((tpl["nt"], TILE, TILE), np.nan, "float32"); full[ids] = mm[h, k]
                a = from_tiles(full, hgt, wid); valid = np.isfinite(a)
                fn = os.path.join(d_out, f"{c}_{v}_{FC_LABELS[h]}.tif")
                with rasterio.open(fn, "w", **prof) as dst:
                    dst.write(np.where(valid, np.round(a, 4), NODATA).astype("float32"), 1)
                    dst.set_band_description(1, f"{v} {FC_LABELS[h]} forecast ({scen})")
                    dst.update_tags(county=c, index=v, month=FC_LABELS[h], lead=h + 1, scenario=scen, forecast_origin=MONTHS[-1], model_trained=cfg["trained"],
                                    quantity="forecast index value (climatology + predicted anomaly)")
                row = dict(scenario=scen, county=c, index=v, month=FC_LABELS[h], lead=h + 1, file=fn, valid_share=float(valid.mean()),
                           mean=float(a[valid].mean()) if valid.any() else np.nan, min=float(a[valid].min()) if valid.any() else np.nan, max=float(a[valid].max()) if valid.any() else np.nan)
                pv = a[::max(1, hgt // 500), ::max(1, wid // 500)].copy() if (v == PRIMARY and h in (0, H - 1)) else None
                return row, ((scen, c, h), pv) if pv is not None else None
            with ThreadPoolExecutor(TIFF_WORKERS) as ex:
                for row, pv in ex.map(write_one, [(k, h) for k in range(len(VIS)) for h in range(H)]):
                    rows_.append(row)
                    if pv: preview[pv[0]] = pv[1]
            print(f"  {c} / {scen}: {len(VIS) * H} files written to {d_out}", flush=True)
            del mm
    return pd.DataFrame(rows_), preview

TIFFS, PREVIEW = write_geotiffs() if RUN_FULL_AREA else (pd.DataFrame(), {})
if len(TIFFS):
    TIFFS.to_csv(os.path.join(OUT_DIR, "forecast_geotiff_index.csv"), index=False)
    print(f"\n{len(TIFFS)} GeoTIFFs. Valid share of pixels, lowest and mean over files:", TIFFS.groupby("county")["valid_share"].agg(["min", "mean"]).round(3).to_dict("index"))
    scen0 = CENTRAL if CENTRAL in FULL_SCENARIOS else FULL_SCENARIOS[0]
    fig, axes = plt.subplots(len(COUNTIES), 2, figsize=(9, 3.8 * len(COUNTIES)), squeeze=False)
    for ci, c in enumerate(COUNTIES):
        for j, h in enumerate((0, H - 1)):
            a = PREVIEW.get((scen0, c, h))
            if a is None: continue
            im = axes[ci, j].imshow(a, cmap="RdYlGn", vmin=0, vmax=0.9); axes[ci, j].set_title(f"{c}: {PRIMARY} forecast {FC_LABELS[h]} ({scen0})"); axes[ci, j].axis("off"); plt.colorbar(im, ax=axes[ci, j], shrink=.8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "forecast_geotiff_preview.png"), dpi=130); plt.show()
    if DELETE_TILE_CACHE:
        import shutil; shutil.rmtree(TILE_CACHE, ignore_errors=True); print("tile cache removed")

## Checks before using the forecast
1. Rebuilt scalers match the saved ones (sections 1-2).
2. No forecast month missing in any scenario (section 3).
3. Rain ratios near 1 and small temperature gaps in the weather table (section 4).
4. Tile self-check passes (section 9).
5. `normal` and `mean` almost equal = little weather signal.
6. Do not present index/lead pairs with skill at or below 0 as forecasts.
7. The first forecast map should look like Sep 2026, with no tile edges.

In [ ]:
# Impact test for the Kakamega RH mismatch: how much do the forecasts change when the model is fed
# the tile-path inputs (weather from the coarse chain) instead of the stored sample inputs (weather from the npz)?
# Run in Notebook C after sections 1-9 have run. Read-only; takes about a minute.

c = "Kakamega"; ci = COUNTIES.index(c)
idx = np.flatnonzero(CID == ci)
pts = Pts(COORDS[0][idx], COORDS[1][idx], TPL[c]["crs"])
bins = np.digitize(ST[idx, ELEV_COL], P["edges"][ci])

# inputs the way the tiles build them (encoder + decoder from the coarse chain)
hw = points_weather(c, ci, pts, DZ[idx], bins, ["mean"])
XE_t, STZ_t, _, ok_t = build_inputs(VI[idx], QA[idx], ci, bins, ST[idx], hw)
xd_t, wok_t = decoder_core(hw["fc"]["mean"], hw["wclim"][:, CAL_FUT, :], np.full(len(idx), ci), "mean", AN_OFF["mean"])

# inputs the way the sample path builds them (stored npz weather)
xd_s, wok_s = decoder_inputs("mean")
a_t = predict(np.nan_to_num(XE_t), xd_t, STZ_t).mean(0) * P["an_sd"]            # [n, H, K] anomalies, index units
a_s = predict(np.nan_to_num(XE[idx]), xd_s[idx], STZ[idx]).mean(0) * P["an_sd"]

ok = ok_t & ENC_OK[idx] & wok_t & wok_s[idx]
d = np.abs(a_t - a_s)[ok]                                                       # [m, H, K]
ref = np.abs(a_s[ok])
print(f"{c}: {ok.sum()} pixels compared")
print(pd.DataFrame(dict(mean_abs_anomaly=ref.mean((0, 1)), mean_abs_diff=d.mean((0, 1)), p99_diff=np.percentile(d, 99, axis=(0, 1)),
                        diff_as_share_of_anomaly=d.mean((0, 1)) / ref.mean((0, 1))), index=VIS).round(4).to_string())

k = VIS.index("NDVI")
q = pd.qcut(ST[idx, ELEV_COL][ok], 5, duplicates="drop")
print("\nNDVI mean abs difference by elevation quintile (index units):")
print(pd.Series(d[:, :, k].mean(1)).groupby(q, observed=True).agg(["size", "mean", "max"]).round(4).to_string())
print("\nNDVI mean abs difference by lead month:", dict(zip(FC_LABELS, d[:, :, k].mean(0).round(4))))

In [ ]:
# Diagnostic cell for Notebook C: run AFTER the self-check cell (needs COARSE, WX, DZ, ST, P, TPL, ... in the kernel)
# Question: why does the tile weather differ from the sample weather for Kakamega?
import numpy as np, pandas as pd

def diagnose(c, tol_dz=0.5, n_worst=8):
    ci = COUNTIES.index(c)
    idx = np.flatnonzero(CID == ci)
    pts = Pts(COORDS[0][idx], COORDS[1][idx], TPL[c]["crs"])
    print(f"\n================ {c}: {len(idx)} sample pixels ================")

    # ---- A. dz: npz (used by the check) vs raster (used by the real tile run) ----
    dz_npz = DZ[idx]
    dz_tif = sample_tif_nearest(TPL[c]["dz"], pts)
    both = np.isfinite(dz_npz) & np.isfinite(dz_tif)
    dd = dz_tif - dz_npz
    print(f"A. dz  NaN in npz: {(~np.isfinite(dz_npz)).sum()} | NaN in raster: {(~np.isfinite(dz_tif)).sum()}")
    print(f"   |raster - npz|: mean {np.nanmean(np.abs(dd)):.3f} m, max {np.nanmax(np.abs(dd)):.2f} m, "
          f"pixels > {tol_dz} m: {(np.abs(dd[both]) > tol_dz).sum()}")
    el = ST[idx, ELEV_COL]
    q = pd.qcut(el, 5, duplicates="drop")
    print("   mean |dz diff| by elevation quintile:")
    print(pd.Series(np.abs(dd)).groupby(q, observed=True).agg(["mean", "max"]).round(3).to_string())

    # ---- B. elevation bins: npz elevation vs static raster elevation ----
    with rasterio.open(TPL[c]["static"]) as s:
        x2, y2 = pts.in_crs(s.crs); cols, rows = (~s.transform) * (x2, y2)
        cols, rows = np.floor(cols).astype(int), np.floor(rows).astype(int)
        band = s.read(1)
    ok = (rows >= 0) & (rows < band.shape[0]) & (cols >= 0) & (cols < band.shape[1])
    el_tif = np.full(len(idx), np.nan, "float32"); el_tif[ok] = band[rows[ok], cols[ok]]
    # NOTE: the tile code uses the 'elev' channel of the static stack; band 1 is used here as in tile_ids/draw_random_pixels
    b_npz = np.digitize(el, P["edges"][ci]); b_tif = np.digitize(el_tif, P["edges"][ci])
    print(f"B. elevation: pixels outside static raster: {(~ok).sum()} | "
          f"|raster - npz| mean {np.nanmean(np.abs(el_tif - el)):.2f} m, max {np.nanmax(np.abs(el_tif - el)):.1f} m | "
          f"pixels whose elevation bin changes: {(b_npz != b_tif).sum()}")

    # ---- C. raw weather: npz WX vs chain_at on coarse_hist ----
    ch = chain_at(COARSE[(c, "hist")], pts, dz_npz, list(range(M)))     # what the tile code rebuilds
    ref = WX[idx].astype("float32")                                       # what Notebook A stored
    d = ch - ref
    print("C. raw weather difference (rebuilt - stored in npz), per variable:")
    rows_ = []
    for k, w in enumerate(WXN):
        dk = d[..., k]
        rows_.append(dict(var=w, nan_stored=int((~np.isfinite(ref[..., k])).sum()), nan_rebuilt=int((~np.isfinite(ch[..., k])).sum()),
                          mean_abs=np.nanmean(np.abs(dk)), max_abs=np.nanmax(np.abs(dk)),
                          frac_gt_1e3=float((np.abs(dk) > 1e-3).mean())))
    print(pd.DataFrame(rows_).round(4).to_string(index=False))

    # month pattern: constant offset every month (stale/changed coarse_hist) vs only some months
    bad_m = np.nanmax(np.abs(d), axis=(0, 2))
    print("   max |diff| by month (months with diff > 1e-3):",
          {MONTHS[i]: round(float(bad_m[i]), 3) for i in np.flatnonzero(bad_m > 1e-3)} or "none")
    # does the difference scale with dz? (would point at lapse rate / dz)
    kT = WXN.index("temp"); dT = d[..., kT].mean(1)
    if np.isfinite(dT).sum() > 10 and np.nanstd(dz_npz) > 0:
        slope = np.polyfit(dz_npz[np.isfinite(dT)], dT[np.isfinite(dT)], 1)[0]
        print(f"   temp diff vs dz slope: {slope:.4f} K per m (manifest lapse = {LAPSE_T})")

    # ---- D. is it a few pixels (grid edge) or all of them? ----
    pix_bad = np.nanmax(np.abs(d), axis=(1, 2)) > 1e-3
    print(f"D. pixels with any raw-weather diff > 1e-3: {pix_bad.sum()} of {len(idx)}")
    if pix_bad.any():
        # distance of the pixel from the coarse grid edge, in coarse cells (temp grid)
        v = "temp"; cx, cy = pts.in_crs(COARSE[(c, "hist")].crs[v])
        cf, rf = (~COARSE[(c, "hist")].tf[v]) * (cx, cy)
        h, w = COARSE[(c, "hist")].arr[v].shape[1:]
        edge = np.minimum.reduce([cf, rf, w - cf, h - rf])
        print(f"   distance to coarse-grid edge (cells): bad pixels median {np.median(edge[pix_bad]):.2f}, "
              f"others median {np.median(edge[~pix_bad]):.2f}; min over bad {edge[pix_bad].min():.2f}")
        w_ix = np.argsort(-np.nanmax(np.abs(d), axis=(1, 2)))[:n_worst]
        print(pd.DataFrame(dict(x=COORDS[0][idx][w_ix].round(0), y=COORDS[1][idx][w_ix].round(0), elev=el[w_ix].round(0),
                                dz=dz_npz[w_ix].round(1), edge_cells=edge[w_ix].round(2),
                                max_diff=np.nanmax(np.abs(d), axis=(1, 2))[w_ix].round(3))).to_string(index=False))

    # ---- E. stored coarse files vs time ----
    import os, time
    f = MAN["counties"][c]["coarse_hist"]
    print(f"E. {os.path.basename(os.path.dirname(f))}/coarse_hist.npz modified:", time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(f))),
          "| training npz modified:", time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(DATA_NPZ))))

diagnose("Kakamega")
diagnose("Kilifi")      # control: should show ~0 everywhere

In [ ]:
`